# Part 14 (실습) — Human-in-the-loop: 발행 전 승인

> **이 노트북의 목표**
> Part 13의 문서 그래프에 **발행 전 승인 노드**를 추가함. `interrupt`로 "발행할까요?"에서 멈추고, `Command(resume=...)`로 승인/수정/반려를 받아 재개함. 같은 thread_id로 멈춤→재개가 이어지는 것을 확인함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~13 · 네 신호의 마지막 ④사람 개입

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai langgraph

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.3)
print("준비 완료")

## 1. 상태와 노드 — 초안 → (승인 대기) → 발행

Part 13 구조에 "발행 전 승인" 단계를 넣음. 발행은 **비가역 행동**이라 사람 확인이 필요(14-1).

In [ ]:
from typing import TypedDict, Annotated
import operator

class State(TypedDict):
    topic: str
    draft: str
    final: str                              # 승인/수정된 최종본
    history: Annotated[list, operator.add]

def draft_node(state: State) -> dict:
    """초안 작성 (되돌릴 수 있음 → 개입 불필요)"""
    draft = model.invoke(f"'{state[\'topic\']}' 주제로 홍보 문구 한 줄만 써줘.").content.strip()
    return {"draft": draft, "history": [f"초안: {draft}"]}

def publish_node(state: State) -> dict:
    """발행 (비가역 → 앞에 승인 노드를 둠)"""
    return {"history": [f"📤 발행됨: {state[\'final\']}"]}

print("상태·노드 정의 완료")

## 2. interrupt — 승인 노드에서 멈추기

`interrupt(...)`로 그래프를 멈추고 사람에게 보여줄 정보를 넘김. 반환값은 재개 시 채워짐(14-2).

In [ ]:
from langgraph.types import interrupt, Command

def approval_node(state: State) -> dict:
    """발행 전 사람 승인 — 여기서 그래프가 멈춤"""
    decision = interrupt({
        "question": "이 문구를 발행할까요?",
        "draft": state["draft"],
        "options": ["approve(승인)", "edit(수정)", "reject(반려)"],
    })
    # ↓ decision은 Command(resume=...)로 재개할 때 채워짐 (14-3)
    if decision["type"] == "approve":
        return {"final": state["draft"], "history": ["✅ 사람 승인"]}
    elif decision["type"] == "edit":
        return {"final": decision["text"], "history": [f"✏️ 사람 수정: {decision[\'text\']}"]}
    else:  # reject
        return {"final": "", "history": ["❌ 사람 반려 → 발행 안 함"]}

print("승인 노드 정의 — interrupt로 멈춤")

## 3. 그래프 조립 — 체크포인터 필수

`interrupt`로 멈추려면 상태를 얼려둘 체크포인터가 반드시 필요함(Part 13). 반려면 발행 건너뛰도록 분기.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

def route_after_approval(state: State) -> str:
    return "publish" if state["final"] else "skip"   # 반려(final 빈값)면 발행 건너뜀

builder = StateGraph(State)
builder.add_node("draft", draft_node)
builder.add_node("approval", approval_node)
builder.add_node("publish", publish_node)
builder.add_node("skip", lambda s: {"history": ["발행 취소됨"]})

builder.add_edge(START, "draft")
builder.add_edge("draft", "approval")
builder.add_conditional_edges("approval", route_after_approval,
                              {"publish": "publish", "skip": "skip"})
builder.add_edge("publish", END)
builder.add_edge("skip", END)

graph = builder.compile(checkpointer=InMemorySaver())   # ← 멈춤·재개의 토대
print("그래프 컴파일 완료 (체크포인터 장착)")

## 4. 실행 → interrupt에서 멈춤

처음 invoke하면 승인 노드의 `interrupt`에서 멈춤. 결과에 인터럽트 정보가 담겨 옴.

In [ ]:
config = {"configurable": {"thread_id": "doc-1"}}   # 멈춤·재개에 같은 thread_id

result = graph.invoke({"topic": "가을 신상 니트", "draft": "", "final": "", "history": []}, config=config)

# 멈춘 지점의 인터럽트 정보 확인
print("그래프가 멈춤. 사람에게 보여줄 정보:")
print(result["__interrupt__"][0].value)

> 📌 `__interrupt__`에 우리가 `interrupt(...)`에 넘긴 정보(질문·초안·옵션)가 담겨 옴. 앱은 이걸 사람에게 보여주고 결정을 받음.

## 5. Command(resume=...)로 재개 — 승인/수정/반려

**같은 thread_id**로 `Command(resume=결정)`을 넘기면 멈췄던 지점부터 재개됨(14-3). resume 값이 `interrupt`의 반환값이 됨.

In [ ]:
# (A) 승인 — 그대로 발행
cfg_a = {"configurable": {"thread_id": "approve-1"}}
graph.invoke({"topic": "가을 니트", "draft": "", "final": "", "history": []}, config=cfg_a)  # 멈춤
r = graph.invoke(Command(resume={"type": "approve"}), config=cfg_a)                          # 재개
print("[승인]", r["history"][-2:])

In [ ]:
# (B) 수정 — 사람이 고친 문구로 발행
cfg_b = {"configurable": {"thread_id": "edit-1"}}
graph.invoke({"topic": "가을 니트", "draft": "", "final": "", "history": []}, config=cfg_b)
r = graph.invoke(Command(resume={"type": "edit", "text": "포근함을 입다, 가을 캐시미어"}), config=cfg_b)
print("[수정]", r["history"][-2:])

In [ ]:
# (C) 반려 — 발행 안 함
cfg_c = {"configurable": {"thread_id": "reject-1"}}
graph.invoke({"topic": "가을 니트", "draft": "", "final": "", "history": []}, config=cfg_c)
r = graph.invoke(Command(resume={"type": "reject"}), config=cfg_c)
print("[반려]", r["history"][-2:])

> 📌 같은 그래프인데 사람의 결정(approve/edit/reject)에 따라 다르게 흐름. 멈춤→재개에 **같은 thread_id**를 쓴 게 핵심 — 다르면 상태가 복원 안 됨.

## 6. 전체 흐름 한눈에

In [ ]:
print(graph.get_graph().draw_ascii())

## ✏️ 미니 실습

**과제**: "비가역적이지 않은" 초안 작성 노드(`draft`)에는 왜 interrupt를 두지 않았는지 한 줄로 설명해보고(주석), 만약 발행 금액이 큰 경우에만 승인을 받게 하려면 어디를 바꿔야 할지 생각해보기.

In [ ]:
# TODO: 주석으로 답 작성
# 1) draft에 interrupt를 안 둔 이유:
#    →
# 2) "고액일 때만 승인"으로 바꾸려면:
#    → route 함수/조건 분기로 "위험하면 approval 노드로, 아니면 바로 publish로" (14-5 차등 개입)

<details><summary>👉 정답</summary>

```python
# 1) draft는 되돌릴 수 있는 행동(초안은 다시 쓰면 됨)이라 개입 불필요. 
#    개입은 비가역·고위험 행동(발행)에만 둠 (14-1, 14-5).
# 2) draft 다음에 조건 분기를 두어:
#    "금액 > 기준 → approval 노드 / 아니면 → 바로 publish"
#    즉 개입조차 조건부로 둠 (14-5 차등 개입).
```
</details>

## 정리

| 절 | 한 일 | API |
|---|---|---|
| 1~2 | 승인 노드 | `interrupt(...)` |
| 3 | 체크포인터 | `compile(checkpointer=...)` |
| 4 | 멈춤 | invoke → `__interrupt__` |
| 5 | 재개 | `Command(resume=...)`, 같은 thread_id |
| 6 | 세 패턴 | 승인/수정/반려 분기 |

### 3줄 요약
1. `interrupt`로 비가역 행동(발행) 직전에 멈추고, 체크포인터가 상태를 얼림.
2. `Command(resume=값)` + 같은 thread_id로 멈춘 지점부터 재개.
3. 승인·수정·반려는 모두 interrupt 반환값을 보고 분기하는 것.

### ▶ 다음 (Part 15)
한 에이전트로 벅찬 일을 여러 전문 에이전트로 나누는 **멀티에이전트 오케스트레이션** — 감독자(supervisor)가 작업을 분배함.